# SpoofCloudFinder Dataset Profile

This notebook profiles the combined April-May 2026 Parquet dataset stored in this repository. It focuses on dataset size, schema, missing values, label coverage, numeric summaries, and example rows.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

DATA_DIR = Path("Datasets")
DATASET_NAME = "April-May 2026.parquet"
DATASET_PATH = DATA_DIR / DATASET_NAME
COLUMN_ALIASES = {
    "timestamp": ("timestamp",),
    "mmsi": ("mmsi",),
    "lat": ("lat",),
    "lon": ("lon",),
    "sog": ("sog",),
    "cog": ("cog",),
    "y_true": ("y_true",),
}
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Expected combined dataset at {DATASET_PATH}")
DATASET_PATH

PosixPath('Datasets/April-May 2026.parquet')

In [2]:
def first_existing_column(df: pd.DataFrame, aliases: tuple[str, ...], target: str) -> pd.Series:
    for column_name in aliases:
        if column_name in df.columns:
            return df[column_name]
    raise KeyError(f"Missing required column for {target}: expected one of {aliases}")


def to_canonical_frame(df: pd.DataFrame) -> pd.DataFrame:
    canonical_df = pd.DataFrame(
        {
            target: first_existing_column(df, aliases, target)
            for target, aliases in COLUMN_ALIASES.items()
        }
    ).copy()
    canonical_df["timestamp"] = pd.to_datetime(canonical_df["timestamp"])
    return canonical_df


def summarise_frame(name: str, df: pd.DataFrame) -> dict:
    labeled_mask = df["y_true"].notna()
    return {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "time_min": df["timestamp"].min(),
        "time_max": df["timestamp"].max(),
        "unique_mmsi": df["mmsi"].nunique(),
        "labeled_rows": int(labeled_mask.sum()),
        "labeled_rate_pct": round(labeled_mask.mean() * 100, 4),
        "positive_labels": int((df["y_true"] == True).sum()),
        "negative_labels": int((df["y_true"] == False).sum()),
        "unlabeled_rows": int((~labeled_mask).sum()),
    }

In [3]:
raw_dataset = pd.read_parquet(DATASET_PATH)
dataset = to_canonical_frame(raw_dataset)
datasets = {DATASET_NAME: dataset}
overview_df = pd.DataFrame([summarise_frame(DATASET_NAME, dataset)]).reset_index(drop=True)
overview_df

,dataset,rows,columns,time_min,time_max,unique_mmsi,labeled_rows,labeled_rate_pct,positive_labels,negative_labels,unlabeled_rows
0,April-May 2026.parquet,2909832,7,2026-04-01 00:00:00.127201,2026-05-31 23:59:57,14373,2909832,100.0,261627,2648205,0


In [4]:
reference_name, reference_df = next(iter(datasets.items()))
reference_columns = list(reference_df.columns)
reference_dtypes = [str(dtype) for dtype in reference_df.dtypes]

print(f"Schema for {reference_name}:")
schema_df = pd.DataFrame(
    {
        "column": reference_columns,
        "dtype": reference_dtypes,
    }
)
schema_df

Schema for April-May 2026.parquet:


,column,dtype
0,timestamp,datetime64[ns]
1,mmsi,str
2,lat,float64
3,lon,float64
4,sog,float64
5,cog,float64
6,y_true,bool


In [5]:
null_frames = []
for name, df in datasets.items():
    null_counts = df.isna().sum()
    null_frames.append(
        pd.DataFrame(
            {
                "dataset": name,
                "column": null_counts.index,
                "null_count": null_counts.values,
                "null_pct": (null_counts.values / len(df) * 100).round(4),
            }
        )
    )

null_summary_df = pd.concat(null_frames, ignore_index=True)
null_summary_df = null_summary_df.sort_values(["dataset", "null_count"], ascending=[True, False]).reset_index(drop=True)
null_summary_df

,dataset,column,null_count,null_pct
0,April-May 2026.parquet,cog,44054,1.5140
1,April-May 2026.parquet,sog,32757,1.1257
2,April-May 2026.parquet,timestamp,0,0.0000
3,April-May 2026.parquet,mmsi,0,0.0000
4,April-May 2026.parquet,lat,0,0.0000
5,April-May 2026.parquet,lon,0,0.0000
6,April-May 2026.parquet,y_true,0,0.0000


In [6]:
label_frames = []
numeric_frames = []

for name, df in datasets.items():
    label_frames.append(
        pd.DataFrame(
            {
                "dataset": [name],
                "positive_labels": [int((df["y_true"] == True).sum())],
                "negative_labels": [int((df["y_true"] == False).sum())],
                "unlabeled_rows": [int(df["y_true"].isna().sum())],
            }
        )
    )

    numeric_df = df[["lat", "lon", "sog", "cog"]].describe().transpose().reset_index()
    numeric_df = numeric_df.rename(columns={"index": "column"})
    numeric_df.insert(0, "dataset", name)
    numeric_frames.append(numeric_df)

label_summary_df = pd.concat(label_frames, ignore_index=True)
numeric_summary_df = pd.concat(numeric_frames, ignore_index=True)
display(label_summary_df)
numeric_summary_df

,dataset,positive_labels,negative_labels,unlabeled_rows
0,April-May 2026.parquet,261627,2648205,0


,dataset,column,count,mean,std,min,25%,50%,75%,max
0,April-May 2026.parquet,lat,2909832.0,53.474561,9.013270,-90.0,53.355906,55.033102,56.481372,91.00000
1,April-May 2026.parquet,lon,2909832.0,10.330795,27.526761,-180.0,7.114543,11.125473,15.582469,179.99964
2,April-May 2026.parquet,sog,2877075.0,13.178231,19.500742,0.0,1.200000,9.800000,13.900000,333.00000
3,April-May 2026.parquet,cog,2865778.0,191.138721,112.625319,0.0,84.900000,208.300000,286.500000,409.50000


In [7]:
for name, df in datasets.items():
    print()
    print(name)
    display(df.head(3))


April-May 2026.parquet


,timestamp,mmsi,lat,lon,sog,cog,y_true
0,2026-04-01 00:13:19.684929,209313000,54.519893,12.197590,8.5,197.2,False
1,2026-04-01 00:17:04.563994,209313000,54.512357,12.193427,8.6,197.7,False
2,2026-04-01 00:18:57.362438,209313000,54.508568,12.191252,8.7,198.5,False
